# scratch.ipynb: R3, ensembles from scratch
Self-contained: repeats the data setup of `comparison.ipynb`.

In [ ]:
import os, time, platform, pickle, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingRegressor,
                              HistGradientBoostingClassifier)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, log_loss
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold, GridSearchCV,
                                     RandomizedSearchCV, cross_val_score, cross_validate,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings("ignore", category=FutureWarning)   # sklearn >= 1.9 deprecates criterion="friedman_mse" (same as squared_error)
SEED = 42
QUICK = os.environ.get("QUICK") == "1"   # smoke-test mode only; leave unset for the real run

# ---- data: Telco Customer Churn (IBM sample data) ----
if os.path.exists("Telco-Customer-Churn.csv"):
    df = pd.read_csv("Telco-Customer-Churn.csv")
else:
    from sklearn.datasets import fetch_openml
    df = fetch_openml(data_id=42178, as_frame=True).frame
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(), errors="coerce")
X = df.drop(columns=["customerID", "Churn"], errors="ignore")   # 11 NaNs stay; imputed inside pipelines
y = (df["Churn"].astype(str) == "Yes").astype(int)
num = X.select_dtypes("number").columns.tolist()
cat = X.columns.difference(num).tolist()

# ---- metric: recall at the top 10% of scores ----
def recall_at_top10(y_true, y_score, frac=0.10):
    y_true = np.asarray(y_true); n = len(y_true); k = int(np.ceil(frac * n))
    rng = np.random.default_rng(0)
    order = np.lexsort((rng.random(n), -np.asarray(y_score)))
    return y_true[order[:k]].sum() / y_true.sum()

def prep(scale):
    steps = [("imp", SimpleImputer(strategy="constant", fill_value=0))]
    if scale: steps.append(("sc", StandardScaler()))
    return ColumnTransformer([("num", Pipeline(steps), num),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False), cat)])

idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.2, stratify=y, random_state=SEED)
X_tr, X_te = X.iloc[idx_tr], X.iloc[idx_te]
y_tr, y_te = y.iloc[idx_tr].values, y.iloc[idx_te].values
pp = prep(scale=False).fit(X_tr)                       # preprocessing fitted on train only
A_tr, A_te = pp.transform(X_tr), pp.transform(X_te)
print(A_tr.shape, A_te.shape)

## Random forest from DecisionTreeClassifier, with own OOB estimate

In [ ]:
class MyRandomForest:
    def __init__(self, n_estimators=300, max_features="sqrt", min_samples_leaf=5, seed=0):
        self.n_estimators, self.max_features = n_estimators, max_features
        self.min_samples_leaf, self.seed = min_samples_leaf, seed
    def fit(self, A, y):
        rng, n = np.random.default_rng(self.seed), len(y)
        self.trees, inbag = [], []
        for _ in range(self.n_estimators):
            idx = rng.integers(0, n, n)                                   # bootstrap sample
            t = DecisionTreeClassifier(max_features=self.max_features,    # random subset at EVERY split
                                       min_samples_leaf=self.min_samples_leaf,
                                       random_state=int(rng.integers(2**31 - 1)))
            t.fit(A[idx], y[idx])
            m = np.zeros(n, bool); m[idx] = True
            self.trees.append(t); inbag.append(m)
        s, c = np.zeros(n), np.zeros(n)                                   # out-of-bag: rows scored only by trees that never saw them
        for t, m in zip(self.trees, inbag):
            s[~m] += t.predict_proba(A[~m])[:, 1]; c[~m] += 1
        assert (c > 0).all()
        self.oob_proba_ = s / c
        return self
    def predict_proba(self, A):
        return np.mean([t.predict_proba(A)[:, 1] for t in self.trees], axis=0)

N_TREES = 60 if QUICK else 300
mine = MyRandomForest(N_TREES).fit(A_tr, y_tr)
sk = RandomForestClassifier(n_estimators=N_TREES, max_features="sqrt", min_samples_leaf=5, bootstrap=True,
                            oob_score=True, random_state=0, n_jobs=-1).fit(A_tr, y_tr)
print(f"TEST recall@10%   scratch RF: {recall_at_top10(y_te, mine.predict_proba(A_te)):.3f} | "
      f"sklearn RF: {recall_at_top10(y_te, sk.predict_proba(A_te)[:, 1]):.3f}")
print(f"OOB  recall@10%   scratch RF: {recall_at_top10(y_tr, mine.oob_proba_):.3f} | "
      f"sklearn RF: {recall_at_top10(y_tr, sk.oob_decision_function_[:, 1]):.3f}")

**Why they differ:** the bootstrap draws and per-split feature subsets come from different random streams, so the individual trees differ and the two averages differ by Monte Carlo noise that shrinks as the number of trees grows.

## Gradient boosting (squared loss) from DecisionTreeRegressor, regression target = MonthlyCharges

In [ ]:
class MyGBR:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.M, self.lr, self.d = n_estimators, learning_rate, max_depth
    def fit(self, X, y):
        self.f0 = float(np.mean(y))                                       # initial constant
        F, self.trees = np.full(len(y), self.f0), []
        for _ in range(self.M):
            t = DecisionTreeRegressor(max_depth=self.d, criterion="friedman_mse", random_state=0)
            t.fit(X, y - F)                                               # fit the residuals
            F += self.lr * t.predict(X); self.trees.append(t)
        return self
    def predict(self, X):
        F = np.full(X.shape[0], self.f0)
        for t in self.trees: F += self.lr * t.predict(X)
        return F

Xr_df = pd.get_dummies(df.drop(columns=["customerID", "Churn", "MonthlyCharges"], errors="ignore"),
                       drop_first=True).astype(float).fillna(0)
# identical or complementary columns (e.g. the R1 "no internet service" group) create exact split ties that sklearn and
# this loop may break differently: same train predictions, different test predictions. Drop them before comparing.
def _partition(c):                       # a binary column and its complement give the same split
    return c if (c.nunique() > 2 or c.iloc[0] == 0) else 1 - c
Xr_df = Xr_df.loc[:, ~Xr_df.apply(_partition).T.duplicated()]
print("regression design:", Xr_df.shape)
Xr = Xr_df.values
yr = df["MonthlyCharges"].values
Xr_tr, Xr_te, yr_tr = Xr[idx_tr], Xr[idx_te], yr[idx_tr]
LR, DEPTH, ROUNDS = 0.1, 3, 100
g_mine = MyGBR(ROUNDS, LR, DEPTH).fit(Xr_tr, yr_tr)
g_sk = GradientBoostingRegressor(loss="squared_error", criterion="friedman_mse", learning_rate=LR,
                                 max_depth=DEPTH, n_estimators=ROUNDS, subsample=1.0, random_state=0).fit(Xr_tr, yr_tr)
for name, Z in [("train", Xr_tr), ("test", Xr_te)]:
    a, b = g_mine.predict(Z), g_sk.predict(Z)
    print(f"GBR {name}: np.allclose = {np.allclose(a, b)}, max |diff| = {np.abs(a - b).max():.2e}")